# Tokenization Math — Exercises

Hands-on exercises covering BPE, Unigram EM, WordPiece PMI, compression analysis, vocabulary tradeoffs, fertility, Viterbi segmentation, information theory, and multilingual fairness.

**Instructions:** Each exercise has a problem statement, a scaffold cell (your workspace), and a solution cell.

**Exercises:**
1. Manual BPE — character merging and compression
2. Compression Ratio — bits-per-character analysis
3. Vocabulary Size Tradeoff — parameter cost and weight tying
4. Fertility Analysis — cross-language tokenization efficiency
5. Viterbi Segmentation — DP-based optimal tokenization
6. Unigram EM Training — E-step, M-step, pruning with ΔL
7. WordPiece PMI — PMI vs frequency merge selection, `##` encoding
8. Context Window & Multilingual Penalty — effective context by language
9. Digit Tokenization — arithmetic failure modes and positional value
10. Information-Theoretic Deep Dive — Rényi spectrum, H₁−H₂ gap, efficiency
11. Segmentation Counting — Fibonacci connection and growth rates
12. Tokenization Cost Calculator — fairness report card and savings analysis
13. **Bonus:** Token Entropy Calculator

In [ ]:
import numpy as np
np.random.seed(42)
from collections import Counter, defaultdict

---
## Exercise 1: Manual BPE

Starting with corpus `"aabaabaab"` and initial vocabulary `{a, b}`:

1. Count all adjacent pairs in the character-level representation
2. Perform **2 merge steps** (merge the most frequent pair each time)
3. What is the final vocabulary?
4. What is the compression ratio after both merges?

In [ ]:
# Exercise 1 — Your solution
corpus = "aabaabaab"

# Step 1: Start with character-level tokens
tokens = list(corpus)
print(f'Initial tokens: {tokens}')
print(f'Token count: {len(tokens)}')

# TODO: Count adjacent pairs
# pairs = ...

# TODO: Merge step 1 — find and merge the most frequent pair
# ...

# TODO: Merge step 2 — find and merge the next most frequent pair
# ...

# TODO: Print final vocabulary and compression ratio
# compression_ratio = len(corpus) / len(final_tokens)

In [ ]:
# Exercise 1 — Solution

def get_pairs(tokens):
    pairs = Counter()
    for i in range(len(tokens) - 1):
        pairs[(tokens[i], tokens[i+1])] += 1
    return pairs

def merge_pair(tokens, pair, new_token):
    result = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            result.append(new_token)
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result

corpus = "aabaabaab"
tokens = list(corpus)
vocab = set(tokens)

print(f'Initial: {tokens}  (len={len(tokens)})')
print(f'Vocab:   {vocab}\n')

# Step 1: count pairs
pairs = get_pairs(tokens)
print(f'Pair counts: {dict(pairs)}')

# Merge 1: most frequent pair
best = max(pairs, key=pairs.get)
new_tok = best[0] + best[1]
tokens = merge_pair(tokens, best, new_tok)
vocab.add(new_tok)
print(f'\nMerge 1: {best} → "{new_tok}"')
print(f'Tokens:  {tokens}  (len={len(tokens)})')

# Merge 2
pairs = get_pairs(tokens)
print(f'Pair counts: {dict(pairs)}')
best = max(pairs, key=pairs.get)
new_tok = best[0] + best[1]
tokens = merge_pair(tokens, best, new_tok)
vocab.add(new_tok)
print(f'\nMerge 2: {best} → "{new_tok}"')
print(f'Tokens:  {tokens}  (len={len(tokens)})')

print(f'\nFinal vocabulary: {vocab}')
print(f'Compression ratio: {len(corpus)}/{len(tokens)} = {len(corpus)/len(tokens):.2f}')

---
## Exercise 2: Compression Ratio Analysis

A tokenizer with V = 32,000 tokenizes **1 million characters** of English into **280,000 tokens**.

1. What is the compression ratio ρ?
2. How many bits per character does this representation use?
3. English text has entropy ~1.3 bits/char (Shannon). Is the tokenizer near-optimal?

In [ ]:
# Exercise 2 — Your solution

n_chars = 1_000_000
n_tokens = 280_000
V = 32_000

# TODO: Calculate compression ratio
# rho = ...

# TODO: Calculate bits per character
# Hint: each token needs log2(V) bits to represent, then scale by compression
# bits_per_token = ...
# bits_per_char = ...

# TODO: Compare to Shannon entropy of English (~1.3 bits/char)

In [ ]:
# Exercise 2 — Solution

n_chars = 1_000_000
n_tokens = 280_000
V = 32_000

# 1. Compression ratio
rho = n_chars / n_tokens
print(f'1. Compression ratio ρ = {n_chars:,} / {n_tokens:,} = {rho:.2f} chars/token')

# 2. Bits per character
# Each token is one of V choices → needs log2(V) bits
bits_per_token = np.log2(V)
# But each token covers ρ characters on average
bits_per_char = bits_per_token / rho
print(f'\n2. Bits per token = log₂({V:,}) = {bits_per_token:.2f} bits')
print(f'   Bits per char  = {bits_per_token:.2f} / {rho:.2f} = {bits_per_char:.2f} bits/char')

# 3. Comparison to Shannon entropy
shannon_entropy = 1.3  # bits/char for English
overhead = bits_per_char / shannon_entropy
print(f'\n3. Shannon entropy of English ≈ {shannon_entropy} bits/char')
print(f'   Our tokenizer: {bits_per_char:.2f} bits/char')
print(f'   Overhead: {overhead:.1f}x the theoretical minimum')
print(f'   → This is {(overhead-1)*100:.0f}% above optimal')
print(f'   → Typical for BPE tokenizers (real token distributions are not uniform,')
print(f'     so actual entropy per token is lower than log₂(V))')

# Bonus: actual bits if token distribution is non-uniform
# In practice, token entropy H < log2(V), so real bits/char is lower
H_typical_token = 10.0  # typical for well-trained BPE
real_bits_per_char = H_typical_token / rho
print(f'\n   If actual token entropy H ≈ {H_typical_token} bits (typical for BPE):')
print(f'   Real bits/char ≈ {real_bits_per_char:.2f} → closer to Shannon bound')

---
## Exercise 3: Vocabulary Size Tradeoff

For a model with `d_model = 4096` and total params = 7B:

1. Calculate embedding + LM head parameters for V = 32K, 64K, 128K, 256K
2. What fraction of total model params is vocabulary for each?
3. At what point does vocabulary overhead become a concern?
4. How does weight tying change the analysis?

In [ ]:
# Exercise 3 — Your solution

d_model = 4096
total_params = 7_000_000_000
vocab_sizes = [32_000, 64_000, 128_000, 256_000]

# TODO: For each vocab size, compute:
#   - embedding params = V * d_model
#   - LM head params = d_model * V (linear projection to vocabulary)
#   - total vocab params (with and without weight tying)
#   - fraction of total model

# TODO: Print a formatted table

# TODO: At what V does vocab become >10% of model? >20%?

In [ ]:
# Exercise 3 — Solution

d_model = 4096
total_params = 7_000_000_000
vocab_sizes = [32_000, 64_000, 128_000, 256_000]

print(f'd_model = {d_model:,}   Total = {total_params/1e9:.0f}B params\n')

# Without weight tying
print('=== Without Weight Tying (E ≠ W_out) ===')
print(f'{"V":>10} {"Embed":>12} {"LM Head":>12} {"Total":>14} {"% of 7B":>10}')
print('-' * 62)
for V in vocab_sizes:
    embed = V * d_model
    lm_head = d_model * V
    total_v = embed + lm_head
    pct = total_v / total_params * 100
    print(f'{V:>10,} {embed:>12,} {lm_head:>12,} {total_v:>14,} {pct:>9.1f}%')

print()

# With weight tying
print('=== With Weight Tying (E = W_out^T) ===')
print(f'{"V":>10} {"Shared":>14} {"% of 7B":>10}')
print('-' * 38)
for V in vocab_sizes:
    shared = V * d_model
    pct = shared / total_params * 100
    print(f'{V:>10,} {shared:>14,} {pct:>9.1f}%')

# Find threshold for 10% and 20%
print()
for threshold_pct in [10, 20]:
    # Without tying: 2 * V * d = threshold% * total
    V_threshold_no_tie = int(threshold_pct / 100 * total_params / (2 * d_model))
    V_threshold_tied = int(threshold_pct / 100 * total_params / d_model)
    print(f'{threshold_pct}% threshold: V = {V_threshold_no_tie:,} (untied), {V_threshold_tied:,} (tied)')

print(f'\n→ At V=256K untied: vocab is {256000*4096*2/total_params:.0%} of model — very significant!')
print(f'→ Weight tying halves the cost and is standard practice')
print(f'→ Most models choose V in 32K–128K range to balance compression vs parameter cost')

---
## Exercise 4: Fertility Analysis

**Fertility** = average number of tokens per word (or per semantic unit).

1. Implement a function that computes fertility for a given text and tokenizer
2. Compare fertility across languages
3. Analyze what fertility reveals about training data composition

In [ ]:
# Exercise 4 — Your solution

# We'll use our SimpleBPE as a proxy if tiktoken is not available.
# First, define a helper that trains BPE on a given training corpus.

def get_pairs(tokens):
    pairs = Counter()
    for i in range(len(tokens) - 1):
        pairs[(tokens[i], tokens[i+1])] += 1
    return pairs

def merge_pair(tokens, pair, new_token):
    result = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            result.append(new_token)
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result

def train_bpe(text, num_merges):
    tokens = list(text)
    merges = []
    for _ in range(num_merges):
        pairs = get_pairs(tokens)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        new_tok = best[0] + best[1]
        tokens = merge_pair(tokens, best, new_tok)
        merges.append((best, new_tok))
    return merges

def bpe_encode(text, merges):
    tokens = list(text)
    for pair, new_tok in merges:
        tokens = merge_pair(tokens, pair, new_tok)
    return tokens

# TODO: Compute fertility = tokens_per_word
# def compute_fertility(text, merges):
#     words = text.split()
#     total_tokens = sum(len(bpe_encode(w, merges)) for w in words)
#     return total_tokens / len(words)

# TODO: Train BPE on English-heavy corpus, then test fertility on different languages

In [ ]:
# Exercise 4 — Solution

def compute_fertility(text, merges):
    """Average tokens per whitespace-delimited word."""
    words = text.split()
    if not words:
        return 0.0
    word_fertilities = []
    for word in words:
        tokens = bpe_encode(word, merges)
        word_fertilities.append(len(tokens))
    return np.mean(word_fertilities), word_fertilities

# Train BPE tokenizer on English-heavy corpus
training_corpus = (
    "the quick brown fox jumps over the lazy dog " * 50 +
    "machine learning is transforming artificial intelligence " * 30 +
    "the cat sat on the mat and the dog ran in the park " * 40 +
    "deep neural networks learn representations from data " * 25
)

merges = train_bpe(training_corpus, num_merges=50)
print(f'Trained BPE with {len(merges)} merges\n')

# Test on different texts
test_texts = {
    'English (in-domain)':  'the quick brown fox jumps over the lazy dog',
    'English (out-domain)': 'cryptocurrency regulations affect global markets',
    'Spanish':              'el gato rapido salta sobre el perro perezoso',
    'French':               'le renard brun rapide saute par dessus le chien',
    'Technical':            'backpropagation gradient descent optimization loss',
}

print(f'{"Text Type":<25} {"Avg Fertility":>14} {"Words":>8} {"Tokens":>8}')
print('-' * 60)

for name, text in test_texts.items():
    fertility, per_word = compute_fertility(text, merges)
    n_words = len(text.split())
    n_tokens = sum(per_word)
    print(f'{name:<25} {fertility:>14.2f} {n_words:>8} {n_tokens:>8}')

print(f'\n→ In-domain English has lowest fertility (most merges apply)')
print(f'→ Out-of-domain and non-English text fragments more')
print(f'→ This reveals the training data bias of the tokenizer')

# Per-word fertility for English vs non-English
print(f'\nDetailed per-word fertility (English in-domain):')
eng_text = 'the quick brown fox jumps over the lazy dog'
_, eng_per_word = compute_fertility(eng_text, merges)
for word, fert in zip(eng_text.split(), eng_per_word):
    bar = '█' * fert
    print(f'  {word:<12} → {fert} token(s) {bar}')

---
## Exercise 5: Viterbi Segmentation

Given vocabulary with costs:

| Token | Cost (-log P) |
|-------|---------------|
| "a"   | 1.0           |
| "b"   | 1.5           |
| "ab"  | 0.5           |
| "ba"  | 0.8           |

For the string `"abab"`:

1. Enumerate all possible segmentations
2. Find the optimal segmentation using the Viterbi algorithm (DP)
3. Verify that DP gives the same answer as brute force

In [ ]:
# Exercise 5 — Your solution

vocab_costs = {'a': 1.0, 'b': 1.5, 'ab': 0.5, 'ba': 0.8}
text = 'abab'

# TODO: Enumerate all possible segmentations of "abab"
# Hint: use recursion or itertools to find all ways to split the string
#       such that every piece is in vocab_costs

# TODO: Implement Viterbi DP
# dp[j] = min cost to tokenize text[0:j]

# TODO: Compare results

In [ ]:
# Exercise 5 — Solution

vocab_costs = {'a': 1.0, 'b': 1.5, 'ab': 0.5, 'ba': 0.8}
text = 'abab'

# Part 1: Brute-force enumerate all segmentations
def enumerate_segmentations(text, vocab, start=0):
    """Recursively find all valid segmentations."""
    if start == len(text):
        return [[]]
    results = []
    for end in range(start + 1, len(text) + 1):
        piece = text[start:end]
        if piece in vocab:
            for rest in enumerate_segmentations(text, vocab, end):
                results.append([piece] + rest)
    return results

all_segs = enumerate_segmentations(text, vocab_costs)
print(f'All valid segmentations of "{text}":')
min_cost_brute = float('inf')
min_seg_brute = None
for seg in all_segs:
    cost = sum(vocab_costs[t] for t in seg)
    if cost < min_cost_brute:
        min_cost_brute = cost
        min_seg_brute = seg
    print(f'  {seg}  cost = {cost:.1f}')

print(f'\nBrute-force optimal: {min_seg_brute} (cost={min_cost_brute:.1f})')
print(f'Total segmentations: {len(all_segs)}')

# Part 2: Viterbi DP
print(f'\n=== Viterbi DP ===')
n = len(text)
INF = float('inf')
dp = [INF] * (n + 1)
dp[0] = 0.0
backtrack = [-1] * (n + 1)

max_token_len = max(len(t) for t in vocab_costs)

for j in range(1, n + 1):
    for i in range(max(0, j - max_token_len), j):
        substr = text[i:j]
        if substr in vocab_costs:
            candidate = dp[i] + vocab_costs[substr]
            print(f'  dp[{j}]: consider "{substr}" (cost {vocab_costs[substr]:.1f}) '
                  f'→ dp[{i}] + {vocab_costs[substr]:.1f} = {candidate:.1f}'
                  f'{" ← new best" if candidate < dp[j] else ""}')
            if candidate < dp[j]:
                dp[j] = candidate
                backtrack[j] = i

# Reconstruct
tokens = []
j = n
while j > 0:
    i = backtrack[j]
    tokens.append(text[i:j])
    j = i
tokens.reverse()

print(f'\nDP table: {dp}')
print(f'Viterbi optimal: {tokens} (cost={dp[n]:.1f})')

# Part 3: Verify
print(f'\n=== Verification ===')
print(f'Brute force: {min_seg_brute} cost={min_cost_brute:.1f}')
print(f'Viterbi DP:  {tokens} cost={dp[n]:.1f}')
print(f'Match: {tokens == min_seg_brute and abs(dp[n] - min_cost_brute) < 1e-9}')

---
## Exercise 6: Unigram Language Model & EM Training

The Unigram model assigns probabilities to vocabulary tokens and finds the optimal segmentation via the **Expectation-Maximization** algorithm.

Given vocabulary $V$ with probabilities $\{p(t)\}_{t \in V}$:

$$P(\mathbf{x}) = \prod_{i=1}^{k} p(t_i) \quad \text{(independence assumption)}$$

**Tasks:**

1. Initialize a vocabulary with character + bigram tokens and uniform probabilities
2. **E-step**: For each word, find the Viterbi-optimal segmentation under current probabilities
3. **M-step**: Re-estimate $p(t) = \frac{\text{count}(t)}{\sum_{t'} \text{count}(t')}$ from the segmented corpus
4. Run EM for 10 iterations and plot the log-likelihood convergence
5. **Pruning**: Compute $\Delta\mathcal{L}(t)$ for each token — the loss in log-likelihood if token $t$ is removed. Remove the bottom 20% (by $\Delta\mathcal{L}$) and re-run EM. Compare the vocabulary before and after pruning.

In [ ]:
# Exercise 6 — Your solution

from collections import Counter, defaultdict
import numpy as np
import math

corpus_words = ["abab", "abba", "bab", "aab", "ab", "ba", "a", "b"]
word_counts = Counter({"abab": 5, "ab": 10, "ba": 3, "aab": 4, "a": 8, "b": 6, "bab": 2, "abba": 3})

# Step 1: Build initial vocabulary (all characters + all bigrams seen)
# TODO: Create vocab dict {token: probability}
# Hint: extract unique chars and bigrams from corpus_words

# Step 2: E-step — Viterbi segmentation
# TODO: def viterbi_segment(word, vocab_probs):
#     """Return best segmentation of word under current probabilities."""
#     ...

# Step 3: M-step — re-estimate probabilities
# TODO: Collect token counts from all Viterbi segmentations, normalize

# Step 4: Run EM loop for 10 iterations, track log-likelihood
# TODO: ll = sum over words: word_count * log P(best_segmentation)

# Step 5: Pruning — compute ΔL for each token and remove bottom 20%
# TODO: For each token t, compute ΔL(t) = LL_with_t - LL_without_t

In [ ]:
# Exercise 6 — Solution

import math
from collections import Counter, defaultdict
import numpy as np

corpus_words = ["abab", "abba", "bab", "aab", "ab", "ba", "a", "b"]
word_counts = Counter({"abab": 5, "ab": 10, "ba": 3, "aab": 4, "a": 8, "b": 6, "bab": 2, "abba": 3})

# ----- Step 1: Build initial vocabulary -----
# Extract all chars and bigrams from corpus words
all_chars = set()
all_bigrams = set()
for w in corpus_words:
    for ch in w:
        all_chars.add(ch)
    for i in range(len(w) - 1):
        all_bigrams.add(w[i:i+2])

initial_vocab = list(all_chars | all_bigrams)
# Uniform initialization
vocab_probs = {t: 1.0 / len(initial_vocab) for t in initial_vocab}
print(f"Initial vocabulary ({len(initial_vocab)} tokens): {sorted(initial_vocab)}")

# ----- Step 2: Viterbi segmentation -----
def viterbi_segment(word, probs):
    """Find the highest-probability segmentation of 'word' under token probs."""
    n = len(word)
    # dp[j] = max log-probability of segmenting word[0:j]
    dp = [-float('inf')] * (n + 1)
    dp[0] = 0.0
    back = [-1] * (n + 1)
    
    for j in range(1, n + 1):
        for i in range(max(0, j - max(len(t) for t in probs)), j):
            sub = word[i:j]
            if sub in probs and probs[sub] > 0:
                score = dp[i] + math.log(probs[sub])
                if score > dp[j]:
                    dp[j] = score
                    back[j] = i
    
    # Backtrack
    if dp[n] == -float('inf'):
        return None, -float('inf')
    tokens = []
    j = n
    while j > 0:
        i = back[j]
        tokens.append(word[i:j])
        j = i
    tokens.reverse()
    return tokens, dp[n]

# ----- Step 3 & 4: EM loop -----
log_likelihoods = []

for iteration in range(10):
    # E-step: segment all words
    token_counts = defaultdict(float)
    total_ll = 0.0
    
    for word in corpus_words:
        count = word_counts[word]
        seg, log_prob = viterbi_segment(word, vocab_probs)
        if seg is None:
            continue
        total_ll += count * log_prob
        for tok in seg:
            token_counts[tok] += count
    
    log_likelihoods.append(total_ll)
    
    # M-step: re-estimate probabilities
    total_count = sum(token_counts.values())
    for t in vocab_probs:
        vocab_probs[t] = (token_counts.get(t, 0) + 1e-8) / (total_count + 1e-8 * len(vocab_probs))
    
    # Show progress
    if iteration < 3 or iteration == 9:
        seg_example, _ = viterbi_segment("abab", vocab_probs)
        print(f"Iter {iteration}: LL = {total_ll:.2f}  |  'abab' → {seg_example}")

print(f"\n{'Iteration':<12} {'Log-Likelihood':<18}")
print("-" * 30)
for i, ll in enumerate(log_likelihoods):
    print(f"{i:<12} {ll:<18.4f}")

# ----- Step 5: Pruning with ΔL -----
print(f"\n=== Pruning Analysis ===")
print(f"{'Token':<8} {'Count':<10} {'Prob':<10} {'ΔL':<12}")
print("-" * 42)

delta_L = {}
baseline_ll = log_likelihoods[-1]

for token in sorted(vocab_probs.keys()):
    # Temporarily remove token and re-segment
    backup = vocab_probs[token]
    vocab_probs[token] = 0  # effectively remove
    
    test_ll = 0.0
    feasible = True
    for word in corpus_words:
        count = word_counts[word]
        seg, log_prob = viterbi_segment(word, vocab_probs)
        if seg is None:
            feasible = False
            break
        test_ll += count * log_prob
    
    if feasible:
        delta_L[token] = baseline_ll - test_ll  # positive = removing hurts
    else:
        delta_L[token] = float('inf')  # can't remove (needed for coverage)
    
    vocab_probs[token] = backup  # restore
    
    count_val = token_counts.get(token, 0)
    dl_str = f"{delta_L[token]:.4f}" if delta_L[token] != float('inf') else "∞ (required)"
    print(f"{token:<8} {count_val:<10.0f} {backup:<10.6f} {dl_str}")

# Remove bottom 20% by ΔL (excluding required tokens)
removable = {t: dl for t, dl in delta_L.items() if dl != float('inf')}
n_remove = max(1, len(removable) // 5)
to_remove = sorted(removable, key=removable.get)[:n_remove]

print(f"\nRemoving {n_remove} tokens with lowest ΔL: {to_remove}")
for t in to_remove:
    vocab_probs[t] = 0

# Re-run 3 EM iterations after pruning
print("\nPost-pruning EM:")
for iteration in range(3):
    token_counts_post = defaultdict(float)
    total_ll_post = 0.0
    for word in corpus_words:
        count = word_counts[word]
        seg, log_prob = viterbi_segment(word, vocab_probs)
        if seg is None:
            continue
        total_ll_post += count * log_prob
        for tok in seg:
            token_counts_post[tok] += count
    
    total_count_post = sum(token_counts_post.values())
    for t in vocab_probs:
        if vocab_probs[t] > 0:
            vocab_probs[t] = (token_counts_post.get(t, 0) + 1e-8) / (total_count_post + 1e-8 * sum(1 for v in vocab_probs.values() if v > 0))
    
    seg_ex, _ = viterbi_segment("abab", vocab_probs)
    print(f"  Post-prune iter {iteration}: LL = {total_ll_post:.2f}  |  'abab' → {seg_ex}")

active_vocab = [t for t, p in vocab_probs.items() if p > 0]
print(f"\nFinal vocabulary ({len(active_vocab)} tokens): {sorted(active_vocab)}")
print(f"Removed {len(initial_vocab) - len(active_vocab)} tokens via pruning")

---
## Exercise 7: WordPiece PMI Score Comparison

WordPiece selects merges by **Pointwise Mutual Information**, not raw frequency:

$$\text{PMI}(x, y) = \log \frac{P(xy)}{P(x) \cdot P(y)}$$

- **High PMI**: "q" + "u" → "qu" (almost always co-occur)
- **Low PMI**: "t" + "h" → "th" (both are common individually)

**Tasks:**

1. Given a character-level corpus, compute the frequency of every adjacent pair
2. Compute PMI for each pair: $\text{PMI}(x,y) = \log_2 \left(\frac{f(xy) / N_{\text{pairs}}}{(f(x)/N_{\text{chars}}) \cdot (f(y)/N_{\text{chars}})}\right)$
3. Rank pairs by **frequency** (BPE order) vs **PMI** (WordPiece order) — show at least 3 pairs where they disagree
4. Explain intuitively why PMI prefers rare-but-coupled pairs over common-but-independent pairs
5. Implement the `##` prefix encoding used by BERT's WordPiece: given a word and a vocabulary, produce the greedy left-to-right segmentation with `##` prefixes for non-initial tokens

In [ ]:
# Exercise 7 — Your solution

corpus = (
    "the quick brown fox jumps over the lazy dog " * 20 +
    "queen quietly quoted a quaint quarrel " * 15 +
    "the thick thin thread through the thought " * 10
)

# TODO: Tokenize to characters and count all adjacent pairs
# pair_counts = ...

# TODO: Count individual character frequencies
# char_counts = ...

# TODO: Compute PMI for each pair
# pmi = log2( P(xy) / (P(x) * P(y)) )

# TODO: Rank by frequency (BPE) vs PMI (WordPiece) — find disagreements

# TODO: Implement ## prefix encoding
# def wordpiece_encode(word, vocab):
#     """Greedily encode word using vocab, adding ## prefix for non-initial pieces."""
#     ...

In [ ]:
# Exercise 7 — Solution

import math
from collections import Counter

corpus = (
    "the quick brown fox jumps over the lazy dog " * 20 +
    "queen quietly quoted a quaint quarrel " * 15 +
    "the thick thin thread through the thought " * 10
)

chars = list(corpus)
N_chars = len(chars)

# 1. Count all adjacent pairs
pair_counts = Counter()
for i in range(len(chars) - 1):
    pair_counts[(chars[i], chars[i+1])] += 1
N_pairs = sum(pair_counts.values())

# 2. Count individual character frequencies
char_counts = Counter(chars)

# 3. Compute PMI for each pair
pmi_scores = {}
for (x, y), count_xy in pair_counts.items():
    p_xy = count_xy / N_pairs
    p_x = char_counts[x] / N_chars
    p_y = char_counts[y] / N_chars
    pmi_scores[(x, y)] = math.log2(p_xy / (p_x * p_y))

# Rank by frequency (BPE) vs PMI (WordPiece)
by_freq = sorted(pair_counts.keys(), key=lambda p: -pair_counts[p])
by_pmi = sorted(pmi_scores.keys(), key=lambda p: -pmi_scores[p])

freq_rank = {pair: i for i, pair in enumerate(by_freq)}
pmi_rank = {pair: i for i, pair in enumerate(by_pmi)}

print("=== Top 10 by FREQUENCY (BPE would merge first) ===")
print(f"{'Pair':<12} {'Freq':>6} {'Freq Rank':>10} {'PMI':>8} {'PMI Rank':>10}")
print("-" * 50)
for pair in by_freq[:10]:
    x, y = pair
    disp = repr(x) + "+" + repr(y)
    print(f"{disp:<12} {pair_counts[pair]:>6} {freq_rank[pair]+1:>10} {pmi_scores[pair]:>8.3f} {pmi_rank[pair]+1:>10}")

print("\n=== Top 10 by PMI (WordPiece would merge first) ===")
print(f"{'Pair':<12} {'Freq':>6} {'Freq Rank':>10} {'PMI':>8} {'PMI Rank':>10}")
print("-" * 50)
for pair in by_pmi[:10]:
    x, y = pair
    disp = repr(x) + "+" + repr(y)
    print(f"{disp:<12} {pair_counts[pair]:>6} {freq_rank[pair]+1:>10} {pmi_scores[pair]:>8.3f} {pmi_rank[pair]+1:>10}")

# Find biggest disagreements
print("\n=== Biggest Rank Disagreements ===")
print(f"{'Pair':<12} {'Freq Rank':>10} {'PMI Rank':>10} {'Δ Rank':>8}")
print("-" * 44)
disagreements = []
for pair in pair_counts:
    if pair in pmi_rank and pair in freq_rank:
        delta = abs(freq_rank[pair] - pmi_rank[pair])
        disagreements.append((pair, freq_rank[pair]+1, pmi_rank[pair]+1, delta))
disagreements.sort(key=lambda x: -x[3])
for pair, fr, pr, delta in disagreements[:8]:
    x, y = pair
    disp = repr(x) + "+" + repr(y)
    direction = "← PMI prefers" if pr < fr else "← BPE prefers"
    print(f"{disp:<12} {fr:>10} {pr:>10} {delta:>8}  {direction}")

print("\n→ PMI favors rare-but-coupled pairs (e.g., 'q'+'u': rare individually but near-certain co-occurrence)")
print("→ BPE favors common pairs regardless of whether they're merely common individually")

# 5. WordPiece ## prefix encoding
print("\n=== WordPiece ## Prefix Encoding ===")

def wordpiece_encode(word, vocab):
    """Greedy left-to-right encoding with ## prefix for continuation pieces."""
    tokens = []
    start = 0
    while start < len(word):
        end = len(word)
        found = False
        while end > start:
            substr = word[start:end]
            if start > 0:
                substr = "##" + substr
            if substr in vocab:
                tokens.append(substr)
                start = end
                found = True
                break
            end -= 1
        if not found:
            tokens.append("[UNK]")
            start += 1
    return tokens

# Example vocabulary (BERT-like)
sample_vocab = {
    "un", "##happ", "##ily", "##happy", "##ing", "##ly",
    "happy", "happ", "##y", "##i", "##l",
    "play", "##play", "##ed", "##er",
    "the", "##e", "a", "##a", "##n",
    "un", "##able", "re", "##mark",
    "[UNK]",
}

test_words = ["unhappily", "playing", "unhappy", "remarkable", "the"]
for word in test_words:
    tokens = wordpiece_encode(word, sample_vocab)
    print(f"  '{word}' → {tokens}")

---
## Exercise 8: Context Window & Multilingual Penalty

A model with context window $C$ tokens and tokenizer fertility $\rho$ can process:

$$\text{effective characters} = C \times \rho$$

**Tasks:**

1. Build a table: for $C \in \{2048, 4096, 8192, 32768, 131072\}$ and $\rho \in \{1.2, 2.5, 4.0, 5.5\}$, compute the effective character capacity and convert to approximate pages (assuming 2000 chars/page)
2. Calculate the **multilingual penalty**: if English has $\rho = 4.0$ and Thai has $\rho = 1.1$, how many times more tokens does Thai need for the same text? What fraction of the context window does a Thai user effectively get?
3. Compute the "context cost ratio" for 5 languages across 3 models (GPT-4 128K, Claude 200K, Gemini 1M) — how many pages of text fit in each language?
4. Discuss: if you could double $\rho$ for a low-resource language at no other cost, what would the downstream effects be?

In [ ]:
# Exercise 8 — Your solution

import numpy as np

context_windows = [2048, 4096, 8192, 32768, 131072]
fertilities = [1.2, 2.5, 4.0, 5.5]  # Thai, Japanese, English, German (approx)

# TODO: Task 1 — Build table of effective chars and pages
# chars_per_page = 2000
# for C in context_windows:
#     for rho in fertilities:
#         effective_chars = C * rho
#         pages = effective_chars / chars_per_page

# TODO: Task 2 — Multilingual penalty
# english_rho = 4.0
# thai_rho = 1.1
# penalty = english_rho / thai_rho  # how many more tokens Thai needs

# TODO: Task 3 — Cross-model, cross-language context cost table
# models = {"GPT-4": 128000, "Claude": 200000, "Gemini": 1000000}
# languages = {"English": 4.0, "Spanish": 3.2, "Japanese": 1.6, "Thai": 1.1, "Amharic": 0.8}

# TODO: Task 4 — Discussion (write as comments or print statements)

In [ ]:
# Exercise 8 — Solution

import numpy as np

chars_per_page = 2000

# ===== Task 1: Effective context table =====
print("=== Task 1: Effective Context (characters & pages) ===\n")

context_windows = [2048, 4096, 8192, 32768, 131072]
fertilities = {"Thai (ρ=1.2)": 1.2, "Japanese (ρ=2.5)": 2.5, "English (ρ=4.0)": 4.0, "German (ρ=5.5)": 5.5}

header = f"{'Context C':>12}"
for lang in fertilities:
    header += f" | {lang:>20}"
print(header)
print("-" * (14 + 23 * len(fertilities)))

for C in context_windows:
    row = f"{C:>12,}"
    for lang, rho in fertilities.items():
        eff_chars = C * rho
        pages = eff_chars / chars_per_page
        row += f" | {eff_chars:>10,.0f} ({pages:>5.1f}p)"
    print(row)

# ===== Task 2: Multilingual penalty =====
print("\n\n=== Task 2: Multilingual Penalty ===\n")

english_rho = 4.0
thai_rho = 1.1

token_penalty = english_rho / thai_rho
context_fraction = thai_rho / english_rho

print(f"English fertility ρ = {english_rho}")
print(f"Thai fertility    ρ = {thai_rho}")
print(f"\nToken penalty: Thai needs {token_penalty:.1f}× more tokens for the same text")
print(f"Context fraction: Thai user gets {context_fraction:.1%} of the effective context")
print(f"\nFor a 128K context window:")
print(f"  English: {128000 * english_rho:,.0f} chars = {128000 * english_rho / chars_per_page:.0f} pages")
print(f"  Thai:    {128000 * thai_rho:,.0f} chars = {128000 * thai_rho / chars_per_page:.0f} pages")
print(f"  → Thai user fits {128000 * thai_rho / (128000 * english_rho) * 100:.0f}% as much text")

# ===== Task 3: Cross-model, cross-language context cost =====
print("\n\n=== Task 3: Pages of Text by Model × Language ===\n")

models = {"GPT-4 (128K)": 128_000, "Claude (200K)": 200_000, "Gemini (1M)": 1_000_000}
languages = {"English": 4.0, "Spanish": 3.2, "Japanese": 1.6, "Thai": 1.1, "Amharic": 0.8}

header = f"{'Model':<16}"
for lang in languages:
    header += f" | {lang:>10}"
print(header)
print("-" * (18 + 13 * len(languages)))

for model_name, C in models.items():
    row = f"{model_name:<16}"
    for lang, rho in languages.items():
        pages = C * rho / chars_per_page
        if pages >= 1000:
            row += f" | {pages/1000:>8.1f}K"
        else:
            row += f" | {pages:>9.0f}"
    print(row)

# Ratio table
print(f"\n{'Model':<16}", end="")
for lang in languages:
    print(f" | {lang:>10}", end="")
print()
print("-" * (18 + 13 * len(languages)))
for model_name, C in models.items():
    eng_pages = C * languages["English"] / chars_per_page
    row = f"{model_name:<16}"
    for lang, rho in languages.items():
        pages = C * rho / chars_per_page
        ratio = pages / eng_pages
        row += f" | {ratio:>9.0%}"
    print(row)

# ===== Task 4: Discussion =====
print("\n\n=== Task 4: Impact of Doubling ρ for a Low-Resource Language ===")
print("""
If we could double Thai's fertility from ρ=1.1 to ρ=2.2:

1. CONTEXT CAPACITY: Doubles from 70 pages → 140 pages (128K window)
   → Thai users can now fit entire documents that previously didn't fit

2. API COST: Halves the per-conversation token count
   → Same semantic content costs 50% less
   → Makes AI deployment economically viable in Thai markets

3. TRAINING EFFICIENCY: Half the tokens per training example
   → Training on Thai data is 2× faster
   → Same compute budget covers 2× more Thai content

4. QUALITY: Each token now carries more semantic content
   → Model sees larger context per example → better long-range reasoning
   → Less fragmentation of Thai words → more natural representations

5. LATENCY: Fewer tokens = fewer autoregressive steps at inference
   → 2× faster generation for Thai responses

In practice, achieving this requires:
  - Allocating more vocabulary entries to Thai characters/subwords
  - Training on Thai-heavy corpora during BPE training
  - Potentially using language-specific tokenizer adapters
""")

---
## Exercise 9: Digit Tokenization & Arithmetic Failure Modes

Tokenizers split numbers inconsistently:

```
"123"    → ["123"]           (1 token)
"1234"   → ["123", "4"]     (2 tokens — split!)
"12345"  → ["123", "45"]    (2 tokens)
```

The digit "4" in "1234" has positional value $4 \times 10^0$, but the tokenizer gives no signal about its position within the original number.

**Tasks:**

1. Train a BPE tokenizer on a corpus that includes numbers, and observe how it splits various numbers
2. For each number from 1 to 10,000, record the tokenization. Plot the number of tokens vs the number of digits — show the inconsistency
3. Demonstrate the **carry problem**: tokenize "1234 + 5678 = 6912" and show that token boundaries don't align across the three numbers
4. Implement a **digit-aware tokenizer** that forces every digit to be its own token. Compare the sequence length vs standard BPE
5. Analyze: if a model must learn that token "4" means $4 \times 10^0$ after "123" but $4 \times 10^1$ in "42", how many distinct positional contexts must it memorize?

In [ ]:
# Exercise 9 — Your solution

from collections import Counter

# Helper: BPE functions (reuse from earlier)
def get_pairs(tokens):
    return Counter((tokens[i], tokens[i+1]) for i in range(len(tokens) - 1))

def merge_pair(tokens, pair, new_token):
    result, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            result.append(new_token)
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result

# TODO: Task 1 — Train BPE on a corpus with numbers, observe digit splits

# TODO: Task 2 — Tokenize 1..10000, plot tokens vs digits

# TODO: Task 3 — Tokenize "1234 + 5678 = 6912", show misaligned boundaries

# TODO: Task 4 — Digit-aware tokenizer (force each digit = 1 token)

# TODO: Task 5 — Count positional contexts a model must memorize

In [ ]:
# Exercise 9 — Solution

import numpy as np
from collections import Counter

def get_pairs(tokens):
    return Counter((tokens[i], tokens[i+1]) for i in range(len(tokens) - 1))

def merge_pair(tokens, pair, new_token):
    result, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            result.append(new_token)
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result

def train_bpe(text, num_merges):
    tokens = list(text)
    merges = []
    for _ in range(num_merges):
        pairs = get_pairs(tokens)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        new_tok = best[0] + best[1]
        tokens = merge_pair(tokens, best, new_tok)
        merges.append((best, new_tok))
    return merges

def bpe_encode(text, merges):
    tokens = list(text)
    for pair, new_tok in merges:
        tokens = merge_pair(tokens, pair, new_tok)
    return tokens

# ===== Task 1: Train BPE on corpus with numbers =====
print("=== Task 1: BPE Trained on Numbers ===\n")

# Corpus: English text + many numbers
number_corpus = " ".join([
    "the price is 1234 dollars and 56 cents",
    "order number 98765 shipped on 2024 01 15",
    "population 8900000 area 42000 density 210",
    "scores 95 87 92 100 88 76 99 83 91 85",
    "years 2020 2021 2022 2023 2024 2025",
    "coordinates 40 7128 74 0060",
] * 20)

merges = train_bpe(number_corpus, num_merges=60)

test_numbers = ["1", "12", "123", "1234", "12345", "123456", "9876543"]
print(f"{'Number':<12} {'Tokens':<30} {'#Tokens':>8}")
print("-" * 52)
for num in test_numbers:
    toks = bpe_encode(num, merges)
    print(f"{num:<12} {str(toks):<30} {len(toks):>8}")

# ===== Task 2: Tokenize 1..10000, show inconsistency =====
print("\n\n=== Task 2: Token Count vs Digit Count ===\n")

digit_counts = {}  # digit_count -> list of token_counts
for n in range(1, 10001):
    s = str(n)
    n_digits = len(s)
    n_tokens = len(bpe_encode(s, merges))
    if n_digits not in digit_counts:
        digit_counts[n_digits] = []
    digit_counts[n_digits].append(n_tokens)

print(f"{'Digits':<8} {'Min Toks':>9} {'Max Toks':>9} {'Avg Toks':>9} {'Unique':>8} {'Consistent?':>12}")
print("-" * 58)
for nd in sorted(digit_counts.keys()):
    toks = digit_counts[nd]
    unique_vals = len(set(toks))
    consistent = "✓" if unique_vals == 1 else f"✗ ({unique_vals} variants)"
    print(f"{nd:<8} {min(toks):>9} {max(toks):>9} {np.mean(toks):>9.2f} {unique_vals:>8} {consistent:>12}")

print("\n→ If tokenization were consistent, all numbers with the same # digits")
print("  would produce the same # tokens. The variation shows the problem.")

# ===== Task 3: Carry problem =====
print("\n\n=== Task 3: Carry Problem in Addition ===\n")

problems = [
    ("1234", "5678", "6912"),
    ("999", "1", "1000"),
    ("4567", "8901", "13468"),
]

for a, b, c in problems:
    ta = bpe_encode(a, merges)
    tb = bpe_encode(b, merges)
    tc = bpe_encode(c, merges)
    print(f"  {a} + {b} = {c}")
    print(f"    {a:>8} → {ta}")
    print(f"    {b:>8} → {tb}")
    print(f"    {c:>8} → {tc}")
    
    # Check if boundaries align
    lens_a = [len(t) for t in ta]
    lens_b = [len(t) for t in tb]
    lens_c = [len(t) for t in tc]
    aligned = (lens_a == lens_b == lens_c)
    print(f"    Token lengths: {lens_a} vs {lens_b} vs {lens_c} — {'ALIGNED' if aligned else 'MISALIGNED ✗'}")
    print()

print("→ Token boundaries rarely align across operands and result")
print("→ The model must learn to carry across arbitrary token splits")

# ===== Task 4: Digit-aware tokenizer =====
print("\n\n=== Task 4: Digit-Aware Tokenizer ===\n")

def digit_aware_encode(text, merges):
    """Apply BPE merges but never merge digit characters together."""
    tokens = list(text)
    for pair, new_tok in merges:
        # Skip if the merge would combine digits
        if any(c.isdigit() for c in pair[0]) and any(c.isdigit() for c in pair[1]):
            continue
        tokens = merge_pair(tokens, pair, new_tok)
    return tokens

test = "the price is 1234 dollars and 56 cents"
standard = bpe_encode(test, merges)
digit_safe = digit_aware_encode(test, merges)

print(f"Text: '{test}'\n")
print(f"Standard BPE ({len(standard)} tokens):")
print(f"  {standard}\n")
print(f"Digit-aware  ({len(digit_safe)} tokens):")
print(f"  {digit_safe}\n")
print(f"Overhead: {len(digit_safe) - len(standard)} extra tokens ({(len(digit_safe)/len(standard) - 1)*100:.0f}% longer)")

# ===== Task 5: Positional contexts =====
print("\n\n=== Task 5: Positional Contexts to Memorize ===\n")

# For each unique token that contains digit(s), count how many different
# positional values it can represent
token_positions = {}
for n in range(1, 100000):
    s = str(n)
    toks = bpe_encode(s, merges)
    # Reconstruct positions: each token covers certain digit positions
    pos = 0
    for tok in toks:
        n_digits_in_tok = len(tok)
        # The rightmost digit of this token has position (len(s) - pos - n_digits_in_tok) from right
        right_pos = len(s) - pos - n_digits_in_tok
        if tok not in token_positions:
            token_positions[tok] = set()
        token_positions[tok].add((right_pos, n_digits_in_tok))
        pos += n_digits_in_tok

# Show tokens with most position ambiguity
print(f"{'Token':<10} {'# Contexts':>12} {'Sample positions (rightmost digit place, width)'}")
print("-" * 65)
for tok in sorted(token_positions, key=lambda t: -len(token_positions[t]))[:15]:
    contexts = token_positions[tok]
    samples = sorted(contexts)[:5]
    print(f"{tok:<10} {len(contexts):>12}   {samples}")

print(f"\nTotal unique (token, position) pairs the model must learn: {sum(len(v) for v in token_positions.values())}")
print("→ Each pair requires the model to implicitly learn a different positional value")

---
## Exercise 10: Information-Theoretic Deep Dive

Go beyond basic Shannon entropy to build a complete information-theoretic profile of a tokenizer.

**Tasks:**

1. **Bits-per-character**: For a given tokenizer, compute bits/char under both uniform coding ($\log_2 |V| / \rho$) and entropy-optimal coding ($H_{\text{token}} / \rho$). Compare to the Shannon bound (~1.3 bits/char for English)
2. **Rényi spectrum**: Compute $H_\alpha$ for $\alpha \in \{0.5, 1, 2, 5, 10, \infty\}$ and plot the spectrum. Verify the ordering $H_\infty \leq H_2 \leq H_1 \leq H_{0.5}$
3. **H₁ − H₂ gap**: Compute the gap for (a) a uniform distribution, (b) a Zipfian distribution, (c) your BPE token distribution. Interpret each gap
4. **Perplexity and effective vocabulary**: Compute PPL $= 2^{H_1}$ — the "effective" vocabulary size. What fraction of V is actually "used"?
5. **Efficiency ratio**: $\eta = H / H_{\max}$. Below what $\eta$ should you consider pruning the vocabulary?

In [ ]:
# Exercise 10 — Your solution

import numpy as np
from collections import Counter

# TODO: Task 1 — bits-per-char under uniform vs entropy-optimal coding
# Given: a tokenized corpus (token list), vocab size V, fertility ρ
# Compute: bits/char (uniform) = log2(V) / ρ
#          bits/char (entropy) = H_token / ρ
# Compare to Shannon bound ≈ 1.3 bits/char

# TODO: Task 2 — Rényi spectrum
# def renyi_entropy(probs, alpha):
#     """Compute Rényi entropy of order alpha."""
#     ...
# Compute for alpha = [0.5, 1, 2, 5, 10, inf]

# TODO: Task 3 — H1 - H2 gap for uniform, Zipfian, and BPE distributions

# TODO: Task 4 — Perplexity = 2^H1, effective vocab fraction

# TODO: Task 5 — Efficiency ratio η = H / H_max, pruning threshold

In [ ]:
# Exercise 10 — Solution

import numpy as np
from collections import Counter
import math

# ---------- Helper: BPE for generating token distributions ----------
def get_pairs(tokens):
    return Counter((tokens[i], tokens[i+1]) for i in range(len(tokens) - 1))

def merge_pair(tokens, pair, new_token):
    result, i = [], 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
            result.append(new_token)
            i += 2
        else:
            result.append(tokens[i])
            i += 1
    return result

def train_bpe(text, num_merges):
    tokens = list(text)
    merges = []
    for _ in range(num_merges):
        pairs = get_pairs(tokens)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        tokens = merge_pair(tokens, best, best[0]+best[1])
        merges.append((best, best[0]+best[1]))
    return merges

def bpe_encode(text, merges):
    tokens = list(text)
    for pair, new_tok in merges:
        tokens = merge_pair(tokens, pair, new_tok)
    return tokens

# ---------- Rényi entropy function ----------
def renyi_entropy(probs, alpha):
    """Compute Rényi entropy of order alpha.
    
    Special cases:
        alpha → 1:   Shannon entropy
        alpha = 2:   Collision entropy
        alpha → ∞:   Min-entropy
    """
    probs = probs[probs > 0]  # remove zeros
    
    if alpha == 1:
        # Shannon entropy (limit as α → 1)
        return -np.sum(probs * np.log2(probs))
    elif alpha == np.inf:
        # Min-entropy
        return -np.log2(np.max(probs))
    else:
        return (1 / (1 - alpha)) * np.log2(np.sum(probs ** alpha))

# ---------- Generate BPE token distribution ----------
corpus = (
    "the quick brown fox jumps over the lazy dog " * 50 +
    "machine learning is transforming artificial intelligence " * 30 +
    "deep neural networks learn representations from data " * 25 +
    "natural language processing uses transformers and attention " * 20 +
    "the cat sat on the mat " * 40
)
merges = train_bpe(corpus, num_merges=80)
tokens = bpe_encode(corpus, merges)

counts = Counter(tokens)
total = sum(counts.values())
V = len(counts)
probs = np.array([c / total for c in counts.values()])
rho = len(corpus) / len(tokens)

print(f"Corpus: {len(corpus)} chars → {len(tokens)} tokens")
print(f"Vocabulary size V = {V}")
print(f"Fertility ρ = {rho:.2f} chars/token\n")

# ===== Task 1: Bits-per-character =====
print("=" * 60)
print("Task 1: Bits-Per-Character Analysis")
print("=" * 60)

H_token = renyi_entropy(probs, alpha=1)
bits_per_char_uniform = np.log2(V) / rho
bits_per_char_entropy = H_token / rho
shannon_bound = 1.3

print(f"\n  log₂(V) = log₂({V}) = {np.log2(V):.2f} bits/token")
print(f"  H_token = {H_token:.2f} bits/token")
print(f"  ρ = {rho:.2f} chars/token")
print(f"\n  Uniform coding:  {np.log2(V):.2f} / {rho:.2f} = {bits_per_char_uniform:.2f} bits/char")
print(f"  Entropy coding:  {H_token:.2f} / {rho:.2f} = {bits_per_char_entropy:.2f} bits/char")
print(f"  Shannon bound:   {shannon_bound:.2f} bits/char")
print(f"\n  Uniform overhead:  {bits_per_char_uniform / shannon_bound:.1f}× Shannon bound")
print(f"  Entropy overhead:  {bits_per_char_entropy / shannon_bound:.1f}× Shannon bound")

# ===== Task 2: Rényi spectrum =====
print(f"\n{'=' * 60}")
print("Task 2: Rényi Entropy Spectrum")
print("=" * 60)

alphas = [0.5, 1, 2, 5, 10, np.inf]
alpha_labels = ["0.5", "1 (Shannon)", "2 (Collision)", "5", "10", "∞ (Min-entropy)"]

print(f"\n  {'α':<20} {'H_α (bits)':<14} {'2^H_α (eff. V)':<16}")
print("  " + "-" * 52)
H_values = []
for alpha, label in zip(alphas, alpha_labels):
    H_a = renyi_entropy(probs, alpha)
    H_values.append(H_a)
    eff_V = 2 ** H_a
    print(f"  {label:<20} {H_a:<14.4f} {eff_V:<16.1f}")

# Verify ordering
print(f"\n  Ordering check: H_∞ ≤ H₂ ≤ H₁ ≤ H₀.₅ ≤ H_max")
print(f"  {H_values[-1]:.3f} ≤ {H_values[2]:.3f} ≤ {H_values[1]:.3f} ≤ {H_values[0]:.3f} ≤ {np.log2(V):.3f}")
ordering_holds = all(H_values[i] >= H_values[i+1] - 1e-9 for i in range(len(H_values)-1))
print(f"  Ordering correct: {ordering_holds}")

# ===== Task 3: H₁ − H₂ gap =====
print(f"\n{'=' * 60}")
print("Task 3: H₁ − H₂ Gap Analysis")
print("=" * 60)

# (a) Uniform distribution
n_uniform = 100
probs_uniform = np.ones(n_uniform) / n_uniform
H1_u = renyi_entropy(probs_uniform, 1)
H2_u = renyi_entropy(probs_uniform, 2)

# (b) Zipfian distribution
ranks = np.arange(1, n_uniform + 1)
probs_zipf = (1.0 / ranks)
probs_zipf /= probs_zipf.sum()
H1_z = renyi_entropy(probs_zipf, 1)
H2_z = renyi_entropy(probs_zipf, 2)

# (c) BPE distribution (from above)
H1_b = renyi_entropy(probs, 1)
H2_b = renyi_entropy(probs, 2)

print(f"\n  {'Distribution':<22} {'H₁':>8} {'H₂':>8} {'H₁−H₂':>8} {'Interpretation'}")
print("  " + "-" * 72)
print(f"  {'(a) Uniform (n=100)':<22} {H1_u:>8.3f} {H2_u:>8.3f} {H1_u-H2_u:>8.3f} {'Perfectly balanced'}")
print(f"  {'(b) Zipfian (n=100)':<22} {H1_z:>8.3f} {H2_z:>8.3f} {H1_z-H2_z:>8.3f} {'Heavy tail — many rare tokens'}")
print(f"  {'(c) BPE tokens':<22} {H1_b:>8.3f} {H2_b:>8.3f} {H1_b-H2_b:>8.3f} {'Real tokenizer skew'}")

print(f"\n  Gap interpretation:")
print(f"    ≈ 0:   All tokens well-utilized (uniform-like)")
print(f"    < 0.5: Mild skew — reasonable vocabulary")
print(f"    > 2.0: Heavy tail — vocabulary is wasteful, consider pruning")

# ===== Task 4: Perplexity and effective vocabulary =====
print(f"\n{'=' * 60}")
print("Task 4: Perplexity & Effective Vocabulary")
print("=" * 60)

ppl_uniform = 2 ** H1_u
ppl_zipf = 2 ** H1_z
ppl_bpe = 2 ** H1_b

print(f"\n  {'Distribution':<22} {'V':>6} {'PPL = 2^H₁':>12} {'PPL/V':>8} {'Usage %':>8}")
print("  " + "-" * 60)
print(f"  {'Uniform (n=100)':<22} {n_uniform:>6} {ppl_uniform:>12.1f} {ppl_uniform/n_uniform:>8.1%} {ppl_uniform/n_uniform*100:>7.1f}%")
print(f"  {'Zipfian (n=100)':<22} {n_uniform:>6} {ppl_zipf:>12.1f} {ppl_zipf/n_uniform:>8.1%} {ppl_zipf/n_uniform*100:>7.1f}%")
print(f"  {'BPE (V={V})':<22} {V:>6} {ppl_bpe:>12.1f} {ppl_bpe/V:>8.1%} {ppl_bpe/V*100:>7.1f}%")

print(f"\n  Interpretation: PPL = 'effective vocabulary size'")
print(f"  If PPL << V, most vocabulary entries rarely appear")
print(f"  → Those embeddings are poorly trained and waste parameters")

# ===== Task 5: Efficiency ratio =====
print(f"\n{'=' * 60}")
print("Task 5: Efficiency Ratio η = H / H_max")
print("=" * 60)

H_max = np.log2(V)
eta = H1_b / H_max

print(f"\n  H₁ = {H1_b:.3f} bits")
print(f"  H_max = log₂({V}) = {H_max:.3f} bits")
print(f"  η = {H1_b:.3f} / {H_max:.3f} = {eta:.3f} ({eta*100:.1f}%)")

print(f"\n  Pruning guidelines:")
print(f"    η > 90%:  Excellent — vocabulary is well-utilized")
print(f"    η = 70-90%: Good — some waste but acceptable")
print(f"    η = 50-70%: Moderate — consider pruning rare tokens")
print(f"    η < 50%:  Poor — too many dead tokens, definitely prune")
print(f"\n  Your tokenizer: η = {eta*100:.1f}% → ", end="")
if eta > 0.9:
    print("Excellent utilization")
elif eta > 0.7:
    print("Good utilization")
elif eta > 0.5:
    print("Consider pruning rare tokens")
else:
    print("Vocabulary is wasteful — prune aggressively")

---
## Exercise 11: Segmentation Counting & Fibonacci Connection

For a string of length $n$ with max token length $L$, the number of valid segmentations follows a **generalized Fibonacci** recurrence:

$$C(n) = \sum_{k=1}^{\min(n, L)} C(n-k), \quad C(0) = 1$$

For $L = 2$: $C(n) = C(n-1) + C(n-2) = F_{n+1}$ (Fibonacci).

**Tasks:**

1. Implement `count_segmentations(n, L)` using the recurrence above
2. Verify that for $L = 2$, $C(n) = F_{n+1}$ for $n = 1, \ldots, 20$
3. Build a growth table: $n \in \{5, 10, 15, 20, 30, 50\}$ and $L \in \{2, 3, 4, 8, 16\}$
4. Also implement a brute-force recursive enumerator for small $n$. Time both methods and show the DP speedup
5. What is the growth rate? For $L = 2$ it's $\phi \approx 1.618$ (golden ratio). Estimate the growth rate for $L = 3, 4$ numerically

In [ ]:
# Exercise 11 — Your solution

# TODO: Task 1 — Implement count_segmentations(n, L) using DP
# def count_segmentations(n, L):
#     """Count segmentations of a string of length n with max token length L."""
#     ...

# TODO: Task 2 — Verify C(n) = F_{n+1} for L = 2

# TODO: Task 3 — Growth table for various n and L

# TODO: Task 4 — Brute-force recursive counter, time comparison

# TODO: Task 5 — Estimate growth rates for L = 2, 3, 4

In [ ]:
# Exercise 11 — Solution

import time
import numpy as np

# ===== Task 1: DP segmentation count =====
def count_segmentations(n, L):
    """Count segmentations of length-n string with max token length L.
    
    Recurrence: C(n) = sum_{k=1}^{min(n,L)} C(n-k), C(0) = 1
    """
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        for k in range(1, min(i, L) + 1):
            dp[i] += dp[i - k]
    return dp[n]

# ===== Task 2: Verify Fibonacci connection for L=2 =====
print("=== Task 2: Fibonacci Verification (L=2) ===\n")

def fibonacci(n):
    """Return F_n (0-indexed: F_0=0, F_1=1, F_2=1, F_3=2, ...)"""
    if n <= 0:
        return 0
    a, b = 0, 1
    for _ in range(n - 1):
        a, b = b, a + b
    return b

print(f"{'n':<6} {'C(n, L=2)':<14} {'F_{n+1}':<14} {'Match?'}")
print("-" * 38)
all_match = True
for n in range(1, 21):
    c = count_segmentations(n, 2)
    f = fibonacci(n + 1)
    match = c == f
    all_match = all_match and match
    if n <= 10 or n == 20:
        print(f"{n:<6} {c:<14} {f:<14} {'✓' if match else '✗'}")
print(f"\nAll match: {all_match}")

# ===== Task 3: Growth table =====
print(f"\n\n=== Task 3: Growth Table ===\n")

ns = [5, 10, 15, 20, 30, 50]
Ls = [2, 3, 4, 8, 16]

header = f"{'n':>6}"
for L in Ls:
    header += f" | {'L='+str(L):>14}"
print(header)
print("-" * (8 + 17 * len(Ls)))

for n in ns:
    row = f"{n:>6}"
    for L in Ls:
        c = count_segmentations(n, L)
        if c > 1e12:
            row += f" | {c:>14.2e}"
        else:
            row += f" | {c:>14,}"
        
    print(row)

print(f"\n→ Even for n=20, L=4: {count_segmentations(20, 4):,} possible segmentations")
print(f"→ Viterbi DP finds the optimum in O(n·L) = O({20}·{4}) = {20*4} operations")

# ===== Task 4: Brute-force vs DP timing =====
print(f"\n\n=== Task 4: Brute-Force vs DP Timing ===\n")

def count_brute_force(n, L, start=0):
    """Recursively count all segmentations (exponential time!)."""
    if start == n:
        return 1
    total = 0
    for k in range(1, min(n - start, L) + 1):
        total += count_brute_force(n, L, start + k)
    return total

print(f"{'n':>4} {'L':>3} {'Brute-force':>14} {'DP':>14} {'Speedup':>10} {'Count':>14}")
print("-" * 62)

for n_test, L_test in [(10, 2), (15, 2), (18, 3), (20, 2), (22, 3)]:
    # DP timing
    t0 = time.perf_counter()
    for _ in range(1000):
        c_dp = count_segmentations(n_test, L_test)
    t_dp = (time.perf_counter() - t0) / 1000
    
    # Brute-force timing
    t0 = time.perf_counter()
    c_bf = count_brute_force(n_test, L_test)
    t_bf = time.perf_counter() - t0
    
    speedup = t_bf / t_dp if t_dp > 0 else float('inf')
    assert c_dp == c_bf, f"Mismatch! DP={c_dp}, BF={c_bf}"
    print(f"{n_test:>4} {L_test:>3} {t_bf:>12.6f}s {t_dp:>12.6f}s {speedup:>9.0f}× {c_dp:>14,}")

# ===== Task 5: Growth rate estimation =====
print(f"\n\n=== Task 5: Growth Rate Estimation ===\n")

print(f"{'L':>4} {'Growth rate r':>14} {'Name':<30}")
print("-" * 50)

for L in [2, 3, 4, 8, 16]:
    # Growth rate: C(n) ≈ r^n for large n
    # Estimate r = (C(n+1) / C(n)) for large n
    n_large = 100
    c1 = count_segmentations(n_large, L)
    c2 = count_segmentations(n_large + 1, L)
    r = c2 / c1
    
    name = ""
    if L == 2:
        name = f"φ (golden ratio) ≈ {(1 + 5**0.5)/2:.6f}"
    elif L == 3:
        name = "tribonacci constant"
    elif L == 4:
        name = "tetranacci constant"
    else:
        name = f"→ approaches 2.0"
    
    print(f"{L:>4} {r:>14.6f} {name}")

print(f"\n→ As L → ∞, growth rate → 2.0 (every position is a binary choice: split or don't)")
print(f"→ Golden ratio φ = (1+√5)/2 ≈ {(1+5**0.5)/2:.6f} governs the L=2 case")

---
## Exercise 12: Tokenization Cost Calculator & Fairness Report

Tokenization has direct financial and performance consequences. The **tokenization tax** means non-English languages pay more per conversation.

**Tasks:**

1. Build a cost calculator: given a language's fertility $\rho$, a model's context window $C$, and API price per 1K tokens, compute the cost per conversation (assume 500-char user input + 1000-char model output)
2. Produce a **fairness report card**: for 6 languages × 3 pricing tiers, show (a) cost per conversation, (b) effective context in pages, (c) cost multiplier vs English
3. Calculate **training cost impact**: if training a 7B model on English costs $2M, what does the same semantic content cost in each language?
4. Compute the **context efficiency score**: $\text{CES} = \frac{\rho_{\text{language}}}{\rho_{\text{English}}} \times 100$. Rank languages by CES
5. How much money would a "perfect" multilingual tokenizer save per year for a company processing 10M conversations/month across 6 languages equally?

In [ ]:
# Exercise 12 — Your solution

# Language fertility estimates (chars/token with typical BPE tokenizer)
languages = {
    "English":  4.0,
    "Spanish":  3.2,
    "German":   3.5,
    "Japanese": 1.6,
    "Thai":     1.1,
    "Amharic":  0.8,
}

# TODO: Task 1 — Cost calculator function
# def conversation_cost(rho, C, price_per_1k, user_chars=500, model_chars=1000):
#     """Return cost of one conversation."""
#     ...

# TODO: Task 2 — Fairness report card (6 languages × 3 price tiers)

# TODO: Task 3 — Training cost impact ($2M baseline for English)

# TODO: Task 4 — Context Efficiency Score ranking

# TODO: Task 5 — Annual savings from perfect multilingual tokenizer

In [ ]:
# Exercise 12 — Solution

import numpy as np

# Language fertility estimates (chars/token with typical BPE tokenizer)
languages = {
    "English":  4.0,
    "Spanish":  3.2,
    "German":   3.5,
    "Japanese": 1.6,
    "Thai":     1.1,
    "Amharic":  0.8,
}

# ===== Task 1: Cost calculator =====
def conversation_cost(rho, price_per_1k, user_chars=500, model_chars=1000):
    """Compute cost of one conversation given fertility and pricing.
    
    Args:
        rho: fertility (chars/token)
        price_per_1k: price per 1000 tokens (simplified: same for input/output)
        user_chars: average user input in characters
        model_chars: average model output in characters
    
    Returns:
        (input_tokens, output_tokens, total_tokens, cost)
    """
    input_tokens = user_chars / rho
    output_tokens = model_chars / rho
    total_tokens = input_tokens + output_tokens
    cost = total_tokens / 1000 * price_per_1k
    return input_tokens, output_tokens, total_tokens, cost

print("=== Task 1: Cost Calculator Demo ===\n")
for lang, rho in languages.items():
    in_t, out_t, tot_t, cost = conversation_cost(rho, price_per_1k=0.01)
    print(f"  {lang:<10}: {tot_t:>8.0f} tokens/conv → ${cost:.4f}/conv")

# ===== Task 2: Fairness Report Card =====
print(f"\n\n{'=' * 80}")
print("Task 2: FAIRNESS REPORT CARD")
print(f"{'=' * 80}")

price_tiers = {
    "Budget ($0.002/1K)":  0.002,
    "Standard ($0.01/1K)": 0.01,
    "Premium ($0.06/1K)":  0.06,
}

eng_rho = languages["English"]

for tier_name, price in price_tiers.items():
    print(f"\n--- {tier_name} ---")
    print(f"{'Language':<12} {'ρ':>5} {'Tokens/conv':>12} {'Cost/conv':>12} {'vs English':>12} {'Pages in 128K':>14}")
    print("-" * 70)
    
    _, _, eng_tokens, eng_cost = conversation_cost(eng_rho, price)
    
    for lang, rho in languages.items():
        _, _, tot_t, cost = conversation_cost(rho, price)
        multiplier = cost / eng_cost
        pages = 128_000 * rho / 2000
        flag = "" if multiplier <= 1.5 else " ⚠️" if multiplier <= 2.5 else " 🔴"
        print(f"{lang:<12} {rho:>5.1f} {tot_t:>12.0f} ${cost:>10.4f} {multiplier:>11.1f}× {pages:>13.0f}{flag}")

# ===== Task 3: Training cost impact =====
print(f"\n\n{'=' * 80}")
print("Task 3: TRAINING COST IMPACT")
print(f"{'=' * 80}\n")

base_training_cost = 2_000_000  # $2M for English 7B model

print(f"Baseline: Training 7B model on English = ${base_training_cost:,.0f}")
print(f"(Same semantic content, different token counts due to fertility)\n")

print(f"{'Language':<12} {'ρ':>5} {'Token multiplier':>18} {'Training cost':>16} {'Extra cost':>14}")
print("-" * 68)

for lang, rho in languages.items():
    multiplier = eng_rho / rho  # More tokens needed for same content
    cost = base_training_cost * multiplier
    extra = cost - base_training_cost
    print(f"{lang:<12} {rho:>5.1f} {multiplier:>17.2f}× ${cost:>14,.0f} ${extra:>+13,.0f}")

print(f"\n→ Training on Thai data costs {eng_rho/languages['Thai']:.1f}× more for the same semantic content")
print(f"→ This incentivizes English-heavy training data, perpetuating the cycle")

# ===== Task 4: Context Efficiency Score =====
print(f"\n\n{'=' * 80}")
print("Task 4: CONTEXT EFFICIENCY SCORE (CES)")
print(f"{'=' * 80}\n")

print(f"CES = (ρ_language / ρ_English) × 100")
print(f"A CES of 100 means equal efficiency to English; lower = disadvantaged\n")

ces_scores = {lang: (rho / eng_rho) * 100 for lang, rho in languages.items()}
ranked = sorted(ces_scores.items(), key=lambda x: -x[1])

print(f"{'Rank':>4} {'Language':<12} {'ρ':>5} {'CES':>6} {'Grade':<8} {'Bar'}")
print("-" * 55)
for rank, (lang, ces) in enumerate(ranked, 1):
    if ces >= 90:
        grade = "A"
    elif ces >= 70:
        grade = "B"
    elif ces >= 50:
        grade = "C"
    elif ces >= 30:
        grade = "D"
    else:
        grade = "F"
    bar = "█" * int(ces / 5)
    print(f"{rank:>4} {lang:<12} {languages[lang]:>5.1f} {ces:>5.0f}% {grade:<8} {bar}")

# ===== Task 5: Annual savings from perfect tokenizer =====
print(f"\n\n{'=' * 80}")
print("Task 5: ANNUAL SAVINGS FROM PERFECT MULTILINGUAL TOKENIZER")
print(f"{'=' * 80}\n")

conversations_per_month = 10_000_000
months = 12
# Assume equal distribution across 6 languages
convs_per_lang = conversations_per_month / len(languages)
price_per_1k = 0.01  # standard tier

# Current cost (language-specific fertility)
current_annual = 0
for lang, rho in languages.items():
    _, _, tot_t, cost = conversation_cost(rho, price_per_1k)
    annual_lang = cost * convs_per_lang * months
    current_annual += annual_lang

# Perfect tokenizer: all languages get English-level fertility (ρ=4.0)
perfect_annual = 0
for lang in languages:
    _, _, tot_t, cost = conversation_cost(eng_rho, price_per_1k)  # ρ=4.0 for all
    annual_lang = cost * convs_per_lang * months
    perfect_annual += annual_lang

savings = current_annual - perfect_annual
savings_pct = savings / current_annual * 100

print(f"Scenario: {conversations_per_month/1e6:.0f}M conversations/month, equally distributed")
print(f"          across {len(languages)} languages, at ${price_per_1k}/1K tokens\n")

print(f"  Current annual cost:  ${current_annual:>14,.0f}")
print(f"  Perfect annual cost:  ${perfect_annual:>14,.0f}")
print(f"  Annual savings:       ${savings:>14,.0f} ({savings_pct:.1f}%)")
print(f"\n  Per-language breakdown:")
print(f"  {'Language':<12} {'Current $/yr':>14} {'Perfect $/yr':>14} {'Savings $/yr':>14}")
print("  " + "-" * 58)

for lang, rho in languages.items():
    _, _, _, cost_current = conversation_cost(rho, price_per_1k)
    _, _, _, cost_perfect = conversation_cost(eng_rho, price_per_1k)
    curr = cost_current * convs_per_lang * months
    perf = cost_perfect * convs_per_lang * months
    sav = curr - perf
    print(f"  {lang:<12} ${curr:>13,.0f} ${perf:>13,.0f} ${sav:>13,.0f}")

print(f"\n→ A perfect multilingual tokenizer would save ${savings:,.0f}/year")
print(f"→ Most savings come from the lowest-fertility languages (Thai, Amharic)")
print(f"→ This is why 'tokenizer fairness' is an active research area")

---
## Bonus Exercise: Token Entropy Calculator

Implement a function that takes a tokenized corpus and computes:
1. Shannon entropy $H$
2. Maximum possible entropy $H_{max} = \log_2 |V|$
3. Efficiency $= H / H_{max}$
4. Rényi entropy $H_\alpha$ for $\alpha = 2$

In [ ]:
# Bonus Exercise — Your solution

# TODO: Implement entropy_analysis(tokens) that returns H, H_max, efficiency, H_2
# def entropy_analysis(tokens):
#     ...

# Test with the BPE tokens from Exercise 1

In [ ]:
# Bonus Exercise — Solution

def entropy_analysis(tokens):
    """
    Compute information-theoretic metrics for a token sequence.
    
    Returns dict with:
        H:          Shannon entropy
        H_max:      Maximum entropy (uniform distribution)
        efficiency: H / H_max
        H_2:        Rényi entropy of order 2 (collision entropy)
    """
    counts = Counter(tokens)
    total = sum(counts.values())
    V = len(counts)
    probs = np.array([c / total for c in counts.values()])
    
    # Shannon entropy: H = -sum p log2 p
    H = -np.sum(probs * np.log2(probs))
    
    # Maximum entropy (uniform)
    H_max = np.log2(V) if V > 1 else 0
    
    # Efficiency
    efficiency = H / H_max if H_max > 0 else 0
    
    # Rényi entropy of order 2
    # H_2 = -log2(sum p_i^2)
    H_2 = -np.log2(np.sum(probs ** 2))
    
    return {
        'H': H,
        'H_max': H_max,
        'efficiency': efficiency,
        'H_2': H_2,
        'V': V,
        'N': total,
    }

# Test 1: Uniform distribution (maximum entropy)
uniform_tokens = list('abcdefgh') * 10
result = entropy_analysis(uniform_tokens)
print('=== Uniform Distribution ===')
print(f'V = {result["V"]}, N = {result["N"]}')
print(f'H     = {result["H"]:.3f} bits')
print(f'H_max = {result["H_max"]:.3f} bits')
print(f'Efficiency = {result["efficiency"]:.1%}')
print(f'H₂    = {result["H_2"]:.3f} bits')

print()

# Test 2: Skewed distribution (low entropy)
skewed_tokens = list('a' * 80 + 'b' * 10 + 'c' * 5 + 'd' * 3 + 'e' * 2)
result = entropy_analysis(skewed_tokens)
print('=== Skewed Distribution ===')
print(f'V = {result["V"]}, N = {result["N"]}')
print(f'H     = {result["H"]:.3f} bits')
print(f'H_max = {result["H_max"]:.3f} bits')
print(f'Efficiency = {result["efficiency"]:.1%}')
print(f'H₂    = {result["H_2"]:.3f} bits')

print()

# Test 3: BPE-tokenized text
bpe_corpus = "the cat sat on the mat the cat ate the rat" * 5
merges_test = train_bpe(bpe_corpus, 15)
bpe_tokens = bpe_encode(bpe_corpus, merges_test)
result = entropy_analysis(bpe_tokens)
print('=== BPE Tokenized Text ===')
print(f'V = {result["V"]}, N = {result["N"]}')
print(f'H     = {result["H"]:.3f} bits')
print(f'H_max = {result["H_max"]:.3f} bits')
print(f'Efficiency = {result["efficiency"]:.1%}')
print(f'H₂    = {result["H_2"]:.3f} bits')
print(f'\n→ H₂ ≤ H always (Rényi entropy is a lower bound on Shannon entropy)')
print(f'→ Low efficiency = many rare tokens (Zipf\'s law in action)')

---

**Next:** [Embedding Space Math](../02-Embedding-Space-Math/notes.md) — what happens after tokens become vectors in $\mathbb{R}^d$.